# Structured Outputs

Many LLM applications do not end with text shown to a human. They feed databases, queues, tickets, dashboards, or other functions. That means the model output must be parseable and validated, not merely plausible.

This notebook explores structured outputs by comparing plain prompting with JSON-oriented prompting and validating responses with Pydantic.


## Learning Objectives

By the end of this notebook, you should be able to:

- Define a small extraction schema.
- Call Groq or Mistral for structured ticket extraction.
- Parse and validate model output with Pydantic.
- Distinguish JSON syntax errors from schema errors.
- Design a retry prompt that uses validation feedback.


## Structured Data Context

Structured-output prompting is a bridge between natural language and parsers. The parser diagram below shows a familiar software-engineering pattern: raw text becomes a representation that code can inspect. Prompt engineering for JSON uses the same idea, except the model produces the intermediate structure.

<div align="center">
  <img src="assets/02-parsing-example.png" alt="Parser transformation diagram" width="300" style="max-width: 100%; height: auto;" />
  <p><em>Parser transformation diagram. Source: <a href="https://commons.wikimedia.org/wiki/File:Parsing_Example.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

The abstract syntax tree image makes the validation goal more concrete. Once information is tree-shaped, a schema or Pydantic model can check field names, data types, allowed values, and missing branches more reliably than it can check prose.

<div align="center">
  <img src="assets/02-ast-as-sentence.png" alt="Abstract syntax tree as a sentence" width="700" style="max-width: 100%; height: auto;" />
  <p><em>Abstract syntax tree as a sentence. Source: <a href="https://commons.wikimedia.org/wiki/File:AST_as_a_sentence.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

## Provider And Documentation Notes

Structured output support differs by provider and model. Groq _JSON Object Mode_ and _Schema-oriented Structured Outputs_ have strict schema support only on selected models. Mistral _JSON Mode_ and _Custom Structured Outputs_ also require explicit JSON instructions in the prompt.

To learn more about these, visit:
- [Groq Structured Outputs](https://console.groq.com/docs/structured-outputs)
- [Mistral Structured Outputs](https://docs.mistral.ai/studio-api/conversations/structured-output)


## Shared Provider Helpers


In [ ]:
import json
from typing import Literal, TypedDict

from openai import BadRequestError

from prompt_helpers import chat, client_and_model

## Define The Contract First

The **schema** is the contract between the LLM and the rest of the system. Keep it small at first: fields, valid labels, and types. Add complexity only after you have tests around the current contract.


In [ ]:
from enum import Enum

from pydantic import BaseModel, Field, ValidationError, field_validator


class TicketLabel(str, Enum):
    billing = "BILLING"
    account_access = "ACCOUNT_ACCESS"
    bug_report = "BUG_REPORT"
    feature_request = "FEATURE_REQUEST"
    other = "OTHER"

The Enum constrains the labels before we define the full output object. This avoids mixing vocabulary design with field-level validation.


In [ ]:
class TicketExtraction(BaseModel):
    """Structured representation expected from the model."""

    label: TicketLabel
    priority: Literal["low", "medium", "high"] = Field(
        description="high: customer is blocked, charged twice, or data is at risk; "
        "medium: something is degraded but there is a workaround; "
        "low: questions and requests."
    )
    summary: str = Field(min_length=10, max_length=180)
    confidence: float = Field(ge=0, le=1)

    @field_validator("priority", mode="before")
    @classmethod
    def normalize_priority(cls, value: object) -> object:
        return value.lower().strip() if isinstance(value, str) else value

Pydantic can emit a JSON schema from the model. `Literal` includes the allowed priority values in that schema. We include it in the prompt so the provider sees the same contract Python will enforce.


In [ ]:
schema = TicketExtraction.model_json_schema()
print(json.dumps(schema, indent=2))

## Load Example Messages

These examples are intentionally few so you can inspect every case. In production, the same pattern scales into a larger evaluation set with expected labels and regression checks.


In [ ]:
class ExtractionExample(TypedDict):
    case_id: str
    text: str
    expected_label: TicketLabel
    expected_priority: str

These cases stay inline because the set is small and used only in this notebook. The expected label and priority fields match the validated `TicketExtraction` contract above; free-form summaries and confidence values are inspected rather than asserted exactly.


In [ ]:
examples: list[ExtractionExample] = [
    {
        "case_id": "extract_001",
        "text": "Customer ACME GmbH was charged twice on the March invoice.",
        "expected_label": TicketLabel.billing,
        "expected_priority": "high",
    },
    {
        "case_id": "extract_002",
        "text": "User lost their MFA phone but still has backup codes.",
        "expected_label": TicketLabel.account_access,
        "expected_priority": "medium",
    },
    {
        "case_id": "extract_003",
        "text": "The prediction API times out during overnight batch scoring.",
        "expected_label": TicketLabel.bug_report,
        "expected_priority": "high",
    },
]

print(json.dumps(examples, indent=2))

## Plain Prompting Is Hard To Trust

A plain prompt can ask for JSON, but it does not guarantee that the answer will parse or match the schema. We start there because it shows why validation is needed.


In [ ]:
case = examples[0]
plain_prompt = f"""Extract a support ticket record from this message.
Return JSON with label, priority, summary, and confidence.

Rules:
- label must be one of BILLING, ACCOUNT_ACCESS, BUG_REPORT, FEATURE_REQUEST, OTHER.
- priority must be low, medium, or high.
- summary must be one sentence.
- confidence must be between 0 and 1.

Message: {case["text"]}"""

plain_response = chat(
    [
        {"role": "system", "content": "You extract compact support ticket records."},
        {"role": "user", "content": plain_prompt},
    ],
    temperature=0.1,
)
print(plain_response)

In [ ]:
def parse_ticket(raw_text: str) -> TicketExtraction:
    """Parse JSON text and validate it against the Pydantic model."""

    data = json.loads(raw_text)
    return TicketExtraction.model_validate(data)


try:
    parsed_plain = parse_ticket(plain_response)
    print(parsed_plain.model_dump())
except (json.JSONDecodeError, ValidationError) as error:
    print(type(error).__name__)
    print(error)

In dry-run mode, this error is intentional: the response is wrapped in a markdown code fence, so `json.loads` fails before checking the fields. A live model may instead return bare JSON and pass this first check; plain prompting does not guarantee either outcome.


## JSON Mode With Explicit Instructions

For OpenAI-compatible chat completions, `response_format={"type": "json_object"}` requests JSON Object Mode on providers/models that support it. The prompt still needs to say exactly which JSON object you expect.


In [ ]:
json_prompt = f"""Extract one support ticket record from the message.

Return only a JSON object matching this schema:
{json.dumps(schema, indent=2)}

Rules:
- label must be one of BILLING, ACCOUNT_ACCESS, BUG_REPORT, FEATURE_REQUEST, OTHER.
- priority must be low, medium, or high.
- summary must be one sentence.
- confidence must be between 0 and 1.

Message: {case["text"]}"""

json_response = chat(
    [
        {"role": "system", "content": "You output valid JSON only."},
        {"role": "user", "content": json_prompt},
    ],
    temperature=0.1,
    response_format={"type": "json_object"},
)
print(json_response)

In [ ]:
try:
    parsed_json = parse_ticket(json_response)
    print(parsed_json.model_dump())
except (json.JSONDecodeError, ValidationError) as error:
    print(type(error).__name__)
    print(error)

This time parsing succeeds. JSON Mode returns a bare, valid JSON object that `json.loads` can parse directly, and the schema in the prompt gave the model the exact fields and labels to use.

JSON Mode only guarantees valid **syntax**, not a valid **schema**. A priority value such as `"urgent"` can appear in valid JSON but still fail Pydantic validation, so `parse_ticket` keeps both checks. The next section handles that second kind of error.


## Validation Feedback For Retries

When validation fails, the best approach is to retry the model call rather than guessing a fix yourself. A good retry prompt should tell the model what failed and preserve the original input. Do not silently repair data unless the repair rule is deterministic and documented.


In [ ]:
def build_retry_prompt(
    original_message: str, bad_response: str, error: Exception
) -> str:
    """Create a focused retry prompt from validator feedback."""

    return f"""The previous response did not pass validation.

Original message:
{original_message}

Invalid response:
{bad_response}

Validation error:
{error}

Return only corrected JSON matching the schema:
{json.dumps(schema, indent=2)}"""


bad_response = '{"label": "LOGIN", "priority": "urgent", "summary": "Locked out", "confidence": 1.2}'
try:
    parse_ticket(bad_response)
except (json.JSONDecodeError, ValidationError) as error:
    retry_prompt = build_retry_prompt(case["text"], bad_response, error)
    print(retry_prompt)

## Batch Extraction Pattern

For a real workflow, run extraction over multiple cases, validate each result, and keep the raw response next to the parsed object. That makes failures auditable.


In [ ]:
def extract_ticket(message: str) -> dict[str, object]:
    """Call the model once and return both raw and validated results."""

    prompt = f"""Return a JSON support ticket matching this schema:
{json.dumps(schema, indent=2)}

Message: {message}"""
    try:
        raw = chat(
            [
                {"role": "system", "content": "You output valid JSON only."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
    except BadRequestError as error:
        # Some providers reject JSON Mode output that is not valid JSON with HTTP 400.
        return {"ok": False, "raw": None, "error": str(error)}
    try:
        parsed = parse_ticket(raw)
        return {"ok": True, "raw": raw, "parsed": parsed.model_dump()}
    except (json.JSONDecodeError, ValidationError) as error:
        return {"ok": False, "raw": raw, "error": str(error)}

Run every inline example and compare the validated label and priority to the expected contract fields. Summaries and confidence values remain available for inspection rather than exact-match scoring.


In [ ]:
def check_example(item: ExtractionExample) -> dict[str, object]:
    """Evaluate validated categorical fields against one expected result."""

    result = extract_ticket(item["text"])
    if not result["ok"]:
        return {"case_id": item["case_id"], "matches_expected": False, "result": result}
    parsed = TicketExtraction.model_validate(result["parsed"])
    matches_expected = (
        parsed.label == item["expected_label"]
        and parsed.priority == item["expected_priority"]
    )
    return {
        "case_id": item["case_id"],
        "matches_expected": matches_expected,
        "parsed": parsed.model_dump(),
    }


checked_results = [check_example(item) for item in examples]
print(json.dumps(checked_results, indent=2, default=str))

## Practice

Add a new field named `requires_human_review` (boolean) to the schema. Update the prompt and validator, then run the extractor on three messages.

A good solution should explain when the model should set the value to `true` (e.g. high-risk billing disputes, account lockouts, or unclear messages).


In [ ]:
# @TODO: Add a new field `requires_human_review` to the schema and update prompt
# and validator, then run the extractor on 3 messages.


### Reference Solution

The additional boolean belongs in the validated contract, not only in the prompt text. This version requires review for requests that carry operational risk or cannot be routed confidently.


In [ ]:
class ReviewedTicketExtraction(TicketExtraction):
    """Extraction record with an explicit human-review decision."""

    requires_human_review: bool = Field(
        description="True for high-risk, blocked, or ambiguous support cases."
    )


review_schema = ReviewedTicketExtraction.model_json_schema()
print(json.dumps(review_schema["properties"]["requires_human_review"], indent=2))

A deterministic dry-run fallback keeps this runnable without an API key. The helper below builds that fallback record locally, matching the same fields the model would return live.


In [ ]:
def dry_review_payload(message: str) -> dict[str, object]:
    """Build a deterministic example record when no provider key is set."""

    lowered = message.lower()
    risky = any(word in lowered for word in ["locked", "dispute", "unclear"])
    if any(word in lowered for word in ["charge", "invoice", "refund"]):
        label = "BILLING"
    elif any(word in lowered for word in ["account", "mfa", "login"]):
        label = "ACCOUNT_ACCESS"
    elif any(word in lowered for word in ["add", "setting"]):
        label = "FEATURE_REQUEST"
    else:
        label = "OTHER"
    return {
        "label": label,
        "priority": "medium",
        "summary": message,
        "confidence": 0.72,
        "requires_human_review": risky,
    }

The extractor below sends the new schema to the model, or falls back to the helper above when no provider key is configured. Both live and dry responses pass through the same Pydantic validation step, so the schema remains the enforcement point.


In [ ]:
def extract_review_ticket(message: str) -> ReviewedTicketExtraction:
    """Extract and validate a ticket that includes the review decision."""

    prompt = f"""Return JSON matching this schema:
{json.dumps(review_schema, indent=2)}

Set requires_human_review to true for account lockouts, disputed charges, or unclear routing.
Message: {message}"""
    client, _, _, dry_reason = client_and_model()
    if client is None:
        print(dry_reason)
        raw = json.dumps(dry_review_payload(message))
    else:
        raw = chat(
            [{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
    return ReviewedTicketExtraction.model_validate_json(raw)

In [ ]:
review_messages = [
    "The account is locked after an MFA reset request.",
    "The customer disputes a renewal charge on the invoice.",
    "Please add a dark mode setting to the dashboard.",
]
for message in review_messages:
    try:
        print(extract_review_ticket(message).model_dump())
    except (BadRequestError, ValidationError) as error:
        # One invalid response should not hide the results for the other messages.
        print(f"Extraction failed for {message!r}: {error}")

## Further Reading

- [JSON Schema](https://json-schema.org/learn/getting-started-step-by-step)
- [Pydantic Models](https://docs.pydantic.dev/latest/concepts/models/)
- [Outlines Structured Generation](https://dottxt-ai.github.io/outlines/latest/)
